# Logistic Regression — Version2

Continued feature engineering and optiuna trials to see.

In [1]:
# Install once if needed. Restart the kernel after upgrading scikit-learn.
# %pip install optuna "scikit-learn>=1.8,<1.10" joblib
import numpy as np
import pandas as pd
import optuna
import joblib  
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from preprocessing import import_data, group_diagnoses, group_medications, engineer, features

N_TRIALS = 1
STUDY_NAME = "logistic_f2_simple_v2"
STORAGE = "sqlite:///logistic_regression_Version2_simple.db"

c:\Users\daoke\Documents\GitHub\fall-2026-hospital-readmission-for-diabetes\.venv314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Features

In [2]:
Xt, yt, Xv, yv = import_data()
numeric = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
           "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
visits = ["number_outpatient", "number_emergency", "number_inpatient"]
diags = ["diag_1", "diag_2", "diag_3"]
drugs = Xt.loc[:, "metformin":"metformin-pioglitazone"].columns.tolist()

frames = []
for X in [Xt, Xv]:
    X.drop(columns=["encounter_id", "readmitted"], errors="ignore", inplace=True)
    grouped_diag = group_diagnoses(X)
    grouped_med = group_medications(X)
    for col in diags:
        X[col + "_group"] = grouped_diag[col]
    for col in grouped_med.columns.difference(X.columns):
        X[col] = grouped_med[col]
    engineer(X)
    X["eng_total_visits"] = X[visits].sum(axis=1, min_count=3)
    for col in visits:
        X[col + "_bin"] = pd.cut(X[col], [-1, 0, 1, 2, 4, np.inf],
                                 labels=["0", "1", "2", "3-4", "5+"]).astype(object)
        X[col + "_share"] = X[col] / (1 + X["eng_total_visits"])
    for col in numeric:
        X[col + "_log1p"] = np.log1p(X[col])
        X[col + "_squared"] = X[col] ** 2
    X = X.copy()
    for i, a in enumerate(numeric):
        for b in numeric[i + 1:]:
            X[a + "_x_" + b] = X[a + "_log1p"] * X[b + "_log1p"]
    for col in ["num_procedures", "number_diagnoses"]:
        X[col + "_per_day"] = X[col] / X["time_in_hospital"].clip(lower=1)
    for status in ["No", "Steady", "Up", "Down"]:
        X["eng_drugs_" + status] = X[drugs].eq(status).sum(axis=1)
    X["eng_drugs_active"] = X[drugs].isin(["Steady", "Up", "Down"]).sum(axis=1)
    X["eng_drugs_changed"] = X[drugs].isin(["Up", "Down"]).sum(axis=1)
    X["eng_drugs_missing"] = X[drugs].isna().sum(axis=1)
    X["eng_age_midpoint"] = pd.to_numeric(
        X["age"].str.extract(r"\[(\d+)-", expand=False), errors="coerce") + 5
    for a, b in [("diag_1_group", "a1cresult"), ("diag_1_group", "insulin"),
                 ("age", "discharge_disposition_id"), ("insulin", "change"),
                 ("admission_type_id", "admission_source_id")]:
        X[a + "_with_" + b] = X[a].fillna("Missing").astype(str) + "|" + X[b].fillna("Missing").astype(str)
    for col in diags + ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
        X[col] = X[col].astype(str).where(X[col].notna(), np.nan)
    for col in X.select_dtypes(exclude="number"):
        X[col] = X[col].astype(object).where(X[col].notna(), np.nan)

    frames.append(X)
Xt, Xv = frames

baseline_features = ["diag_1_group" if c == "diag_1" else c for c in features()]
all_features = Xt.columns.tolist()
print(f"{len(all_features)} candidate features")

Importing train/validation data...
Finished the import.


C:\Users\daoke\AppData\Local\Temp\ipykernel_27120\454331577.py:10: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X.drop(columns=["encounter_id", "readmitted"], errors="ignore", inplace=True)
C:\Users\daoke\AppData\Local\Temp\ipykernel_27120\454331577.py:10: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X.drop(columns=["encounter_id", "readmitted"], errors="ignore", inplace=True)


131 candidate features


### One objective

Test a bunch of things.

In [3]:
def objective(trial, return_model=False):
    mode = trial.suggest_categorical("feature_set", ["baseline", "all", "selected"])
    chosen = baseline_features if mode == "baseline" else all_features
    if mode == "selected":
        chosen = [c for c in all_features if trial.suggest_categorical("use_" + c, [True, False])]
    if not chosen:
        return 0.0
    nums = Xt[chosen].select_dtypes(include="number").columns.tolist()
    cats = [c for c in chosen if c not in nums]
    scaler = trial.suggest_categorical("scaler", ["standard", "robust"])
    num = Pipeline([
        ("impute", SimpleImputer(strategy=trial.suggest_categorical("num_impute", ["median", "mean", "constant"]),
                                fill_value=0, keep_empty_features=True,
                                add_indicator=trial.suggest_categorical("missing_indicators", [False, True]))),
        ("scale", StandardScaler() if scaler == "standard" else RobustScaler()),
    ])
    cat = Pipeline([
        ("impute", SimpleImputer(strategy=trial.suggest_categorical("cat_impute", ["constant", "most_frequent"]),
                                fill_value="Missing", keep_empty_features=True)),
        ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=trial.suggest_categorical("min_frequency", [1, 10, 50, 100, 250]))),
    ])
    weight_mode = trial.suggest_categorical("class_weight", ["none", "balanced", "custom"])
    weight = {0: 1, 1: trial.suggest_float("positive_weight", 0.5, 40, log=True)} if weight_mode == "custom" else (
        "balanced" if weight_mode == "balanced" else None)
    ratio = trial.suggest_categorical("l1_ratio", [0.0, 0.25, 0.5, 0.75, 1.0])
    model = Pipeline([
        ("prep", ColumnTransformer([("num", num, nums), ("cat", cat, cats)])),
        ("model", LogisticRegression(C=trial.suggest_float("C", 1e-4, 100, log=True),
            l1_ratio=ratio, solver="lbfgs" if ratio == 0 else "saga",
            class_weight=weight, max_iter=5000, random_state=42)),
    ])
    model.fit(Xt[chosen], yt)
    probabilities = model.predict_proba(Xv[chosen])[:, 1]
    precision, recall, thresholds = precision_recall_curve(yv, probabilities)
    f2 = 5 * precision[:-1] * recall[:-1] / (4 * precision[:-1] + recall[:-1] + 1e-15)
    best = int(f2.argmax())
    trial.set_user_attr("features", chosen)
    trial.set_user_attr("n_features", len(chosen))
    trial.set_user_attr("threshold", float(thresholds[best]))
    if return_model:
        return model, chosen, float(thresholds[best])
    return float(f2[best])

### Runs

Save previous runs and continue runs from before.

In [4]:
study = optuna.create_study(study_name=STUDY_NAME, storage=STORAGE,
                            direction="maximize", load_if_exists=True,
                            sampler=optuna.samplers.TPESampler(n_startup_trials=30))
if not study.trials:
    study.enqueue_trial({"feature_set": "baseline", "C": 0.2, "l1_ratio": 0.0,
                         "class_weight": "custom", "positive_weight": 22.0,
                         "num_impute": "median", "cat_impute": "constant",
                         "missing_indicators": False, "scaler": "standard", "min_frequency": 100})
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, show_progress_bar=True)

[I 2026-10-30 21:02:16,624] Using an existing study with `study_name='logistic_f2_simple_v2'` instead of creating a new one.
Best trial: 437. Best value: 0.430778: 100%|██████████| 1/1 [00:25<00:00, 25.11s/it]

[I 2026-10-30 21:02:42,085] Trial 478 finished with value: 0.4199724042773367 and parameters: {'feature_set': 'selected', 'use_time_in_hospital': False, 'use_num_lab_procedures': True, 'use_num_procedures': False, 'use_num_medications': True, 'use_number_outpatient': False, 'use_number_emergency': True, 'use_number_inpatient': False, 'use_number_diagnoses': False, 'use_race': False, 'use_gender': False, 'use_age': False, 'use_weight': False, 'use_admission_type_id': False, 'use_discharge_disposition_id': True, 'use_admission_source_id': True, 'use_payer_code': False, 'use_medical_specialty': False, 'use_diag_1': True, 'use_diag_2': False, 'use_diag_3': False, 'use_max_glu_serum': False, 'use_a1cresult': True, 'use_metformin': False, 'use_repaglinide': False, 'use_nateglinide': False, 'use_chlorpropamide': False, 'use_glimepiride': True, 'use_acetohexamide': False, 'use_glipizide': False, 'use_glyburide': True, 'use_tolbutamide': True, 'use_pioglitazone': True, 'use_rosiglitazone': Fals

### Best trial and model

In [5]:
print(f"Best validation F2: {study.best_value:.5f}")
print("Threshold:", study.best_trial.user_attrs["threshold"])
print("Parameters:", study.best_params)
print("Features:", study.best_trial.user_attrs["features"])
results = study.trials_dataframe().sort_values("value", ascending=False)
display(results.head(10))
results.to_csv("logistic_regression_Version2_trials.csv", index=False)

best_model, chosen_features, threshold = objective(optuna.trial.FixedTrial(study.best_params), return_model=True)
joblib.dump({"model": best_model, "features": chosen_features, "threshold": threshold,
             "params": study.best_params, "validation_f2": study.best_value},
            "logistic_regression_Version2_best.joblib")

Best validation F2: 0.43078
Threshold: 0.4385136092278073
Parameters: {'feature_set': 'all', 'scaler': 'standard', 'num_impute': 'mean', 'missing_indicators': False, 'cat_impute': 'constant', 'min_frequency': 1, 'class_weight': 'balanced', 'l1_ratio': 0.25, 'C': 0.015731183591896788}
Features: ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'a1cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin

,number,value,datetime_start,datetime_complete,duration,params_C,params_cat_impute,params_class_weight,params_feature_set,params_l1_ratio,...,params_use_tolazamide,params_use_tolbutamide,params_use_troglitazone,params_use_weight,user_attrs_features,user_attrs_n_features,user_attrs_threshold,system_attrs_fixed_params,system_attrs_tpe:relative_params:0,state
437,437,0.430778,2026-10-30 18:52:17.940669,2026-10-30 18:54:28.505307,0 days 00:02:10.564638,0.015731,constant,balanced,all,0.25,...,NaN,NaN,NaN,NaN,"[time_in_hospital, num_lab_procedures, num_pro...",131.0,0.438514,NaN,"{""C"": 0.015731183591896788, ""cat_impute"": ""con...",COMPLETE
454,454,0.430778,2026-10-30 19:33:27.216718,2026-10-30 19:35:39.223783,0 days 00:02:12.007065,0.015729,constant,balanced,all,0.25,...,NaN,NaN,NaN,NaN,"[time_in_hospital, num_lab_procedures, num_pro...",131.0,0.438516,NaN,"{""C"": 0.015728867475404022, ""cat_impute"": ""con...",COMPLETE
412,412,0.430739,2026-10-30 17:16:43.791322,2026-10-30 17:18:53.340089,0 days 00:02:09.548767,0.015509,constant,balanced,all,0.25,...,NaN,NaN,NaN,NaN,"[time_in_hospital, num_lab_procedures, num_pro...",131.0,0.439328,NaN,"{""C"": 0.01550860264410071, ""cat_impute"": ""cons...",COMPLETE
392,392,0.430696,2026-10-30 16:25:28.676972,2026-10-30 16:27:40.041903,0 days 00:02:11.364931,0.015882,constant,balanced,all,0.25,...,NaN,NaN,NaN,NaN,"[time_in_hospital, num_lab_procedures, num_pro...",131.0,0.438326,NaN,"{""C"": 0.015882410596489226, ""cat_impute"": ""con...",COMPLETE
451,451,0.430652,2026-10-30 19:11:35.169146,2026-10-30 19:13:42.310940,0 days 00:02:07.141794,0.015194,constant,balanced,all,0.25,...,NaN,NaN,NaN,NaN,"[time_in_hospital, num_lab_procedures, num_pro...",131.0,0.438842,NaN,"{""C"": 0.015194397762018252, ""cat_impute"": ""con...",COMPLETE
472,472,0.430220,2026-10-30 20:29:30.535471,2026-10-30 20:31:39.253430,0 days 00:02:08.717959,0.016962,constant,balanced,all,0.25,...,NaN,NaN,NaN,NaN,"[time_in_hospital, num_lab_procedures, num_pro...",131.0,0.438419,NaN,"{""C"": 0.016962251729644617, ""cat_impute"": ""con...",COMPLETE
213,213,0.430009,2026-10-29 18:56:28.263551,2026-10-29 19:00:03.646753,0 days 00:03:35.383202,0.048114,constant,balanced,all,1.00,...,NaN,NaN,NaN,NaN,"[time_in_hospital, num_lab_procedures, num_pro...",131.0,0.437069,NaN,"{""C"": 0.04811367840831364, ""cat_impute"": ""cons...",COMPLETE
422,422,0.429978,2026-10-30 17:50:09.473054,2026-10-30 17:52:14.520703,0 days 00:02:05.047649,0.014302,constant,balanced,all,0.25,...,NaN,NaN,NaN,NaN,"[time_in_hospital, num_lab_procedures, num_pro...",131.0,0.440263,NaN,"{""C"": 0.014302278421520266, ""cat_impute"": ""con...",COMPLETE
247,247,0.429896,2026-10-29 21:32:07.048466,2026-10-29 21:36:04.189227,0 days 00:03:57.140761,0.056224,constant,balanced,all,1.00,...,NaN,NaN,NaN,NaN,"[time_in_hospital, num_lab_procedures, num_pro...",131.0,0.426552,NaN,"{""C"": 0.05622377496844962, ""cat_impute"": ""cons...",COMPLETE
450,450,0.429841,2026-10-30 19:09:31.867647,2026-10-30 19:11:35.151518,0 days 00:02:03.283871,0.013939,constant,balanced,all,0.25,...,NaN,NaN,NaN,NaN,"[time_in_hospital, num_lab_procedures, num_pro...",131.0,0.440057,NaN,"{""C"": 0.01393860574158354, ""cat_impute"": ""cons...",COMPLETE


['logistic_regression_Version2_best.joblib']